# Actividad 6: Explorando transformers preentrenados con Hugging Face

**Curso:** Deep Learning  
**Profesor:** Gonzalo A. Ruz  
**Ayudante:** Anthony D. Cho

In [14]:
texts = [
    "I loved this product. It is easy to use and works very well.",
    "This was a disappointing purchase. The quality is poor.",
    "The product is acceptable, but I expected something better."
]

sentiment_results = sentiment_pipeline(texts)
print("Resultados de análisis de sentimiento:")
for i, result in enumerate(sentiment_results):
    print(f"Texto {i+1}: {texts[i]} -> Etiqueta: {result['label']}, Score: {result['score']:.4f}")

positivos = [text for text, res in zip(texts, sentiment_results) if res['label'] == 'POSITIVE']

# Re-evaluating based on the common interpretation: closest score to 0.5 implies highest uncertainty.
# The pipeline output gives a 'score' for the predicted 'label'.
# If 'label' is POSITIVE, 'score' is P(POSITIVE). If 'label' is NEGATIVE, 'score' is P(NEGATIVE).
# Max uncertainty would be when P(POSITIVE) is ~0.5 (and P(NEGATIVE) is ~0.5).
# So, we need to find the text where the score for the predicted label is closest to 0.5.
incertidumbre_text = ""
min_diff_to_0_5 = float('inf')
uncertainty_score = 0.0 # Initialize
for text_item, result_item in zip(texts, sentiment_results):
    score_value = result_item['score']
    diff_to_0_5 = abs(score_value - 0.5)
    if diff_to_0_5 < min_diff_to_0_5:
        min_diff_to_0_5 = diff_to_0_5
        incertidumbre_text = text_item
        uncertainty_score = score_value

Resultados de análisis de sentimiento:
Texto 1: I loved this product. It is easy to use and works very well. -> Etiqueta: POSITIVE, Score: 0.9999
Texto 2: This was a disappointing purchase. The quality is poor. -> Etiqueta: NEGATIVE, Score: 0.9998
Texto 3: The product is acceptable, but I expected something better. -> Etiqueta: NEGATIVE, Score: 0.9093


En esta actividad aplicaremos modelos transformer preentrenados a distintas tareas de procesamiento de lenguaje natural usando `pipeline()`.

Trabajaremos con ejemplos simples y guiados para observar cómo cambian las capacidades del modelo según la tarea.

## Objetivos de la actividad

Al finalizar esta actividad deberías poder:

- usar `pipeline()` para distintas tareas de NLP,
- interpretar salidas de modelos preentrenados,
- comparar distintos usos de transformers,
- reflexionar sobre la diferencia entre modelos clásicos y modelos modernos preentrenados.

## Instrucciones
- La actividad debe ser realizada por los grupos de trabajo
- Responda cada pregunta en las celdas correspondientes
- Justifique brevemente sus respuestas cuando se solicite
- Renombrar el archivo agregando el apellido de las y los integrantes, por ejemplo Actividad6_Tupper_Tudor_Gorosito_Acosta.ipynb
- Subir el archivo al link de entrega Actividad 6 en webcursos que será habilitado
- __Fecha de entrega:__ Idealmente al final del bloque 2 de la clase del 27 de abril 2026. Fecha límite de entrega 4 de mayo 2026

## Integrantes (RUT - Nombre y apellido):

-

-

-

## Instrucciones generales

- Completa las celdas indicadas.
- Lee con atención los comentarios y preguntas.
- Responde las preguntas breves en las celdas Markdown donde se indique.
- Si alguna salida cambia levemente, no te preocupes: eso puede ocurrir según la versión del modelo o del entorno.

In [15]:
!pip -q install transformers

> **Importante:** si Colab pide reiniciar el entorno, hazlo antes de continuar.

## Parte 1: análisis de sentimiento

Comenzaremos con una tarea clásica de NLP: clasificar si un texto expresa un sentimiento positivo o negativo.

Completa la siguiente celda para crear un pipeline de `sentiment-analysis`.

In [16]:
from transformers import pipeline
sentiment_pipeline = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Ahora aplica el modelo a tres textos de ejemplo y observa las predicciones.

In [17]:
# The 'texts' variable is now defined in the previous cell for better execution flow.

### Pregunta 1

Observa las predicciones del modelo.

**Escribe una breve respuesta:**
- ¿Qué textos fueron clasificados como positivos?
- ¿Cuál parece generar más incertidumbre según el score?

**Respuesta:**

- Los textos clasificados como positivos son: `['I loved this product. It is easy to use and works very well.']`
- El texto que parece generar más incertidumbre es: `"The product is acceptable, but I expected something better."`. Su score de `0.9093` para la etiqueta 'NEGATIVE' está más cerca de `0.5` que los scores de los otros textos (que son casi `1.0`), indicando una menor confianza del modelo en su predicción para este texto en comparación con los otros.

## Parte 2: zero-shot classification

Ahora usaremos un pipeline que permite clasificar un texto con etiquetas propuestas por nosotros, sin entrenar un clasificador específico para esas clases.

Completa la siguiente celda con el pipeline adecuado.

In [18]:
zero_shot_pipeline = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [19]:
candidate_labels = ["business", "health", "travel", "food"]

text = "This article reviews the nutritional benefits of eating vegetables, fruits, and whole grains every day."

zero_shot_result = zero_shot_pipeline(text, candidate_labels)
print("Resultados de clasificación zero-shot:")
print(zero_shot_result)

best_label = zero_shot_result['labels'][0]
best_score = zero_shot_result['scores'][0]

Resultados de clasificación zero-shot:
{'sequence': 'This article reviews the nutritional benefits of eating vegetables, fruits, and whole grains every day.', 'labels': ['food', 'health', 'business', 'travel'], 'scores': [0.5119794011116028, 0.48166701197624207, 0.0036916283424943686, 0.00266192015260458]}


### Pregunta 2

Según la salida del modelo:

- ¿cuál fue la etiqueta más probable?
- ¿te parece razonable esa predicción?
- ¿qué ventaja tiene este enfoque frente a entrenar un clasificador desde cero para cada conjunto de etiquetas?

**Respuesta:**

- La etiqueta más probable fue: `food` con un score de `0.5120`.
- Sí, la predicción de `food` es bastante razonable. El texto habla sobre "nutritional benefits of eating vegetables, fruits, and whole grains", lo cual se alinea directamente con el tema de la salud y la alimentación.
- La principal ventaja de este enfoque es que no requiere ningún entrenamiento previo con datos etiquetados para las clases específicas (`candidate_labels`). El modelo utiliza su conocimiento preexistente del lenguaje para determinar la similitud semántica entre el texto de entrada y las etiquetas candidatas. Esto lo hace muy flexible y eficiente para clasificar textos en nuevas categorías sin la necesidad de un gran conjunto de datos de entrenamiento para cada tarea de clasificación.

**Respuesta:**

-
-
-

## Parte 3: question answering

Ahora probaremos una tarea de pregunta-respuesta.

En este caso, el modelo recibe:

- una **pregunta**,
- un **contexto**,

e intenta encontrar la mejor respuesta dentro del texto entregado.

Completa la siguiente celda con el pipeline adecuado.

In [20]:
qa_pipeline = pipeline("question-answering")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [21]:
context = """
The Amazon rainforest is the largest tropical rainforest in the world.
It covers parts of several South American countries, with most of it located in Brazil.
The rainforest is known for its extraordinary biodiversity and plays an important role in regulating the global climate.
"""

question = "Which country contains most of the Amazon rainforest?"

qa_result = qa_pipeline(question=question, context=context)
print("Resultado de pregunta-respuesta:")
print(qa_result)

answer = qa_result['answer']
score = qa_result['score']

Resultado de pregunta-respuesta:
{'score': 0.9890652298927307, 'start': 152, 'end': 158, 'answer': 'Brazil'}


### Pregunta 3

Mira la salida del modelo.

- ¿Qué respuesta encontró?
- ¿Por qué esta tarea es distinta de `text-generation`?

**Respuesta:**

- La respuesta encontrada fue: `"Brazil"` con un score de `0.9891`.
- Esta tarea (`question-answering`) es distinta de `text-generation` porque su objetivo es extraer una respuesta **existente** de un contexto dado, buscando la porción de texto más relevante. En cambio, `text-generation` crea texto **nuevo** y original que no necesariamente está presente en un contexto de entrada, sino que se construye a partir de un `prompt` o inicio, generando secuencias coherentes pero no necesariamente fácticas o directamente extraídas de un documento.

**Respuesta:**

-
-

## Parte 4: prueba tus propios ejemplos

Ahora escribe:

- un texto propio para análisis de sentimiento,
- un texto propio para zero-shot classification,
- una pregunta propia para el contexto dado.

La idea es experimentar un poco con los modelos.

In [22]:
# Escribe aquí un texto propio
my_text_sentiment = "Me encanta cómo funciona este software, es súper intuitivo y me ha ahorrado mucho tiempo."

# Ejecuta el modelo
my_sentiment_result = sentiment_pipeline(my_text_sentiment)
print("Resultado de análisis de sentimiento para tu texto:")
print(my_sentiment_result)


Resultado de análisis de sentimiento para tu texto:
[{'label': 'POSITIVE', 'score': 0.7795957922935486}]


In [23]:
# Escribe aquí tu texto y tus etiquetas
my_text_zero_shot = "Un nuevo estudio sugiere que el consumo de café podría reducir el riesgo de ciertas enfermedades."
my_candidate_labels = ["salud", "ciencia", "negocios", "deportes"]

# Ejecuta el modelo
my_zero_shot_result = zero_shot_pipeline(my_text_zero_shot, my_candidate_labels)
print("Resultado de clasificación zero-shot para tu texto:")
print(my_zero_shot_result)

my_best_label_zero_shot = my_zero_shot_result['labels'][0]
my_best_score_zero_shot = my_zero_shot_result['scores'][0]


Resultado de clasificación zero-shot para tu texto:
{'sequence': 'Un nuevo estudio sugiere que el consumo de café podría reducir el riesgo de ciertas enfermedades.', 'labels': ['salud', 'ciencia', 'negocios', 'deportes'], 'scores': [0.8705666065216064, 0.11755012720823288, 0.007337599061429501, 0.004545754753053188]}


In [24]:
context = """
The Amazon rainforest is the largest tropical rainforest in the world.
It covers parts of several South American countries, with most of it located in Brazil.
The rainforest is known for its extraordinary biodiversity and plays an important role in regulating the global climate.
"""

my_question = "¿Qué es el Amazonas conocido por?"

# Ejecuta el modelo
my_qa_result = qa_pipeline(question=my_question, context=context)
print("Resultado de pregunta-respuesta para tu pregunta:")
print(my_qa_result)

my_answer_qa = my_qa_result['answer']


Resultado de pregunta-respuesta para tu pregunta:
{'score': 0.0354345329105854, 'start': 152, 'end': 158, 'answer': 'Brazil'}


### Pregunta 4

Después de probar tus propios ejemplos, responde brevemente:

- ¿qué tarea te pareció más intuitiva?
- ¿cuál te pareció más sorprendente?
- ¿en cuál confiarías más para una aplicación real?

**Respuesta:**

- La tarea más intuitiva me pareció el **análisis de sentimiento**. Es directo: un texto, una emoción. Mi ejemplo `"Me encanta cómo funciona este software, es súper intuitivo y me ha ahorrado mucho tiempo."` fue clasificado como `POSITIVE` con un score de `{my_sentiment_result[0]['score']:.4f}`, lo cual es muy claro y esperado.
- La más sorprendente fue la **clasificación zero-shot**. Es increíble cómo puede clasificar un texto como `"Un nuevo estudio sugiere que el consumo de café podría reducir el riesgo de ciertas enfermedades."` en la categoría `'{my_best_label_zero_shot}'` con un score de `{my_best_score_zero_shot:.4f}` sin haber sido entrenado específicamente con esas etiquetas. Demuestra una gran comprensión semántica general del modelo.
- Para una aplicación real, confiaría más en el **análisis de sentimiento** y la **clasificación zero-shot** para tareas donde las categorías de salida son claras y el contexto es relativamente directo. La `question-answering` me generó algo de incertidumbre en mi ejemplo `"¿Qué es el Amazonas conocido por?"`, donde el modelo, a pesar de que la respuesta esperada era 'extraordinary biodiversity', respondió `'{my_answer_qa}'` con un score bajo de `{my_qa_result['score']:.4f}`. Esto sugiere que para preguntas más abiertas o que requieren inferencia, el rendimiento puede ser menos robusto si no se ajusta la pregunta o el contexto de forma precisa. En contraste, las clasificaciones fueron más directas y con scores de confianza más altos.

**Respuesta:**

-
-
-

## Parte 5: generación de texto

Prueba una tarea de generación de texto.

Completa la siguiente celda con el pipeline adecuado.

In [25]:
text_gen_pipeline = pipeline("text-generation", model="distilgpt2")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Use su propio texto para generar texto nuevo

In [26]:
prompt = "Once upon a time, in a land far, far away, there was a brave knight who"

generated_text = text_gen_pipeline(prompt, max_new_tokens=40, do_sample=True, temperature=0.7)

print("Texto generado:")
print(generated_text[0]['generated_text'])

generated_output = generated_text[0]['generated_text']

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Texto generado:
Once upon a time, in a land far, far away, there was a brave knight who could hold him against a wall. He had been slain and the gods who ruled his land, which he had known for a long time.


The story of the ancient hero is a tale


### Pregunta 5

¿Qué diferencia observas entre:

- clasificar un texto,
- responder una pregunta sobre un contexto,
- generar texto nuevo?

**Respuesta:**

- **Clasificar un texto (análisis de sentimiento y zero-shot):** Esta tarea toma un texto de entrada y le asigna una etiqueta predefinida (o un conjunto de etiquetas candidatas en zero-shot) basada en su contenido. El modelo no crea texto nuevo, sino que categoriza el existente. Su objetivo es la identificación y categorización de propiedades del texto.
- **Responder una pregunta sobre un contexto (question-answering):** Aquí, el modelo recibe una pregunta y un texto (contexto) y su objetivo es encontrar la parte del texto que mejor responda a la pregunta. Al igual que la clasificación, no genera texto libremente, sino que extrae la respuesta directamente del contexto. La respuesta a mi pregunta "¿Qué es el Amazonas conocido por?" fue "{my_answer_qa}".
- **Generar texto nuevo (text-generation):** Esta tarea es fundamentalmente diferente, ya que el modelo crea texto original a partir de un 'prompt' o inicio. No categoriza ni extrae, sino que inventa nuevas secuencias de palabras que son coherentes y gramaticalmente correctas, continuando la historia o idea propuesta. Por ejemplo, a partir de "{prompt}", el modelo generó: "{generated_output}". Requiere una comprensión profunda del lenguaje para producir contenido que no existía previamente.

## Reflexión final

En la sesión 4 trabajamos con modelos secuenciales como LSTM.
En esta sesión trabajamos con transformers preentrenados.

### Pregunta 6

Escribe una reflexión breve comparando ambos enfoques.

Puedes apoyarte en ideas como:

- entrenamiento desde cero vs modelo preentrenado,
- recurrencia vs atención,
- flexibilidad de tareas,
- facilidad de uso con herramientas modernas.

**Respuesta:**

La comparación entre enfoques con modelos secuenciales (como LSTM) y Transformers preentrenados revela diferencias fundamentales en cómo abordan el procesamiento del lenguaje natural:

- **Entrenamiento:**
    - **LSTM:** Típicamente requieren entrenamiento desde cero para una tarea específica, lo que implica grandes cantidades de datos etiquetados y recursos computacionales considerables.
    - **Transformers preentrenados:** Se benefician de un preentrenamiento masivo en vastos corpus de texto, aprendiendo representaciones ricas del lenguaje. Luego, pueden ser ajustados (fine-tuning) para tareas específicas con relativamente pocos datos, lo que es mucho más eficiente.
- **Arquitectura:**
    - **LSTM (Recurrencia):** Procesan secuencias de forma secuencial, manteniendo un 'estado' oculto que se actualiza a cada paso. Esto puede llevar a problemas de olvido de información a largo plazo y dificultad para paralelizar el entrenamiento.
    - **Transformers (Atención):** Utilizan mecanismos de auto-atención (self-attention) que permiten al modelo ponderar la importancia de diferentes partes de la secuencia de entrada al procesar cada elemento. Esto resuelve el problema de la dependencia a largo plazo y permite un alto grado de paralelización.
- **Flexibilidad de tareas:**
    - **LSTM:** Tienden a ser más específicos para la tarea para la que fueron entrenados (ej. clasificación de sentimientos, traducción). Aunque pueden adaptarse, requieren una re-arquitectura o re-entrenamiento significativo para nuevas tareas.
    - **Transformers preentrenados:** Son extremadamente versátiles y pueden aplicarse a una amplia gama de tareas (clasificación, generación, QA, resumen, etc.) con cambios mínimos en la arquitectura y a menudo solo con fine-tuning, gracias a su rica comprensión del lenguaje.
- **Facilidad de uso con herramientas modernas:**
    - **LSTM:** Requieren más conocimiento de la arquitectura y la implementación manual si no se utilizan frameworks de alto nivel. La gestión de estados y dependencias secuenciales puede ser compleja.
    - **Transformers preentrenados:** Herramientas como `Hugging Face Transformers` (`pipeline()`) han democratizado su uso, permitiendo a los desarrolladores y científicos de datos aplicar modelos de última generación con pocas líneas de código, sin necesidad de entender los detalles internos de la arquitectura o el entrenamiento.

## Resumen Final de Variables

Aquí se presentan las variables clave y sus valores finales tras la ejecución del notebook:

### Parte 1: Análisis de Sentimiento
- **Textos clasificados como positivos (`positivos`):** `{positivos}`
- **Texto con más incertidumbre (`incertidumbre_text`):** `'{incertidumbre_text}'` (Score: `{uncertainty_score:.4f}`)

### Parte 2: Clasificación Zero-Shot
- **Etiqueta más probable (`best_label`):** `'{best_label}'` (Score: `{best_score:.4f}`)

### Parte 3: Question Answering
- **Respuesta encontrada (`answer`):** `'{answer}'` (Score: `{score:.4f}`)

### Parte 4: Ejemplos Propios
- **Resultado de análisis de sentimiento para tu texto (`my_sentiment_result`):** `{'label': '{my_sentiment_result[0]['label']}', 'score': {my_sentiment_result[0]['score']:.4f}}`
- **Etiqueta más probable de clasificación zero-shot (`my_best_label_zero_shot`):** `'{my_best_label_zero_shot}'` (Score: `{my_best_score_zero_shot:.4f}`)
- **Respuesta a tu pregunta (`my_answer_qa`):** `'{my_answer_qa}'` (Score: `{my_qa_result['score']:.4f}`)

### Parte 5: Generación de Texto
- **Prompt inicial (`prompt`):** `'{prompt}'`
- **Texto generado (`generated_output`):** `'{generated_output}'`